# Simple RAG

In this notebook you will index a few short passages with **OpenRouter** embeddings, store them in LangChain's **`InMemoryVectorStore`**, then run a small **LangGraph functional API** workflow: a `@task` retrieves passages, and a second `@task` calls the chat model to produce an answer.

## Environment Setup


### Install dependencies


#### A. on Colab?


In [ ]:
%pip install -q langchain langchain_openai langchain_community

#### B. Locally?


If using `uv` locally you can `uv sync` (provided you have the `pyproject.toml`).


### Environment variables

The following environment variables are used in this notebook:

| Variable               | Required | Purpose                                  |
|------------------------|----------|------------------------------------------|
| `OPENROUTER_API_KEY`   | Yes      | Authenticates requests to OpenRouter (chat and embeddings) |

#### On Colab?

Store these as Colab **Secrets** (key icon in the left sidebar), using the variable names above (for example `OPENROUTER_API_KEY`). Grant this notebook access to the secrets so they can be read securely at runtime.

#### Locally?

Store these in a `.env` file in your project directory (which should be listed in `.gitignore` so it is never committed). The code below uses `python-dotenv` to load values from this file into environment variables at runtime.



::: {.callout-warning}

API Keys are secrets and thus [**shall never be exposed**](./never_expose_api_keys.qmd).

:::


### Reading environment variables in


In [ ]:
import os

In [ ]:
try:
    # In Colab? read from userdata (secrets)
    from google.colab import userdata
    ON_COLAB = True
    os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
except ImportError:
    # Load `.env` file (locally)
    from dotenv import load_dotenv
    load_dotenv(override=True)

In [ ]:
from langchain_openai import ChatOpenAI

# https://openrouter.ai/nvidia/nemotron-3-nano-30b-a3b:free
llm = ChatOpenAI(
    model="nvidia/nemotron-3-nano-30b-a3b:free",
    temperature=0,
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ.get("OPENROUTER_API_KEY"),
)

We use [`@task`](https://docs.langchain.com/oss/python/langgraph/functional-api#task) for steps that call remote models (embedding search and chat), and [`@entrypoint`](https://docs.langchain.com/oss/python/langgraph/functional-api#entrypoint) to compose them. See the M1 lesson for `.stream(..., stream_mode="updates")` and checkpointing options.


In [ ]:
from IPython.display import Markdown, display

from langchain.messages import (
    SystemMessage,
    HumanMessage,
)
from langchain_openai import OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langgraph.func import entrypoint, task

# Embeddings via OpenRouter (OpenAI-compatible API)
# https://openrouter.ai/docs/guides/embeddings
embeddings = OpenAIEmbeddings(
    model="openai/text-embedding-3-small",
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1",
)

## Sample documents

Three short excerpts from a fictional manual:


In [ ]:
DOCUMENT1 = """Operating the Climate Control System  Your Googlecar has a climate control system that allows you to adjust the temperature and airflow in the car. To operate the climate control system, use the buttons and knobs located on the center console.  Temperature: The temperature knob controls the temperature inside the car. Turn the knob clockwise to increase the temperature or counterclockwise to decrease the temperature. Airflow: The airflow knob controls the amount of airflow inside the car. Turn the knob clockwise to increase the airflow or counterclockwise to decrease the airflow. Fan speed: The fan speed knob controls the speed of the fan. Turn the knob clockwise to increase the fan speed or counterclockwise to decrease the fan speed. Mode: The mode button allows you to select the desired mode. The available modes are: Auto: The car will automatically adjust the temperature and airflow to maintain a comfortable level. Cool: The car will blow cool air into the car. Heat: The car will blow warm air into the car. Defrost: The car will blow warm air onto the windshield to defrost it."""
DOCUMENT2 = """Your Googlecar has a large touchscreen display that provides access to a variety of features, including navigation, entertainment, and climate control. To use the touchscreen display, simply touch the desired icon.  For example, you can touch the "Navigation" icon to get directions to your destination or touch the "Music" icon to play your favorite songs."""
DOCUMENT3 = """Shifting Gears Your Googlecar has an automatic transmission. To shift gears, simply move the shift lever to the desired position.  Park: This position is used when you are parked. The wheels are locked and the car cannot move. Reverse: This position is used to back up. Neutral: This position is used when you are stopped at a light or in traffic. The car is not in gear and will not move unless you press the gas pedal. Drive: This position is used to drive forward. Low: This position is used for driving in snow or other slippery conditions."""

documents = [DOCUMENT1, DOCUMENT2, DOCUMENT3]

## Indexing

`InMemoryVectorStore` keeps vectors in process memory—fine for tutorials. Production systems typically use a persisted vector database.


In [ ]:
vectorstore = InMemoryVectorStore.from_texts(
    texts=documents,
    embedding=embeddings,
    ids=[str(i) for i in range(len(documents))],
)
print(f"Indexed {len(vectorstore.store)} documents")

## RAG workflow

`retrieve_passages` runs a similarity search (query embedding + cosine similarity in the store). `generate_answer` sends a `SystemMessage` plus a `HumanMessage` (question and passages) to `llm.invoke`, as in `01_models.ipynb`.


In [ ]:
@task
def retrieve_passages(query: str, k: int = 1) -> list[str]:
    docs = vectorstore.similarity_search(query, k=k)
    return [d.page_content for d in docs]


@task
def generate_answer(query: str, passages: list[str]) -> str:
    query_oneline = query.replace("\n", " ")
    system_text = (
        "You are a helpful and informative assistant. Answer using the reference passages below. "
        "Respond in complete sentences for a non-technical audience in a friendly, conversational tone. "
        "If a passage is irrelevant, you may ignore it."
    )
    passage_block = "\n\n".join(p.replace("\n", " ") for p in passages)
    user_text = f"QUESTION: {query_oneline}\n\nPASSAGES:\n{passage_block}"
    msg = llm.invoke(
        [
            SystemMessage(system_text),
            HumanMessage(user_text),
        ]
    )
    return msg.content


@entrypoint()
def rag_qa_workflow(query: str) -> str:
    passages = retrieve_passages(query).result()
    return generate_answer(query, passages).result()

### Run the workflow

Stream task updates (same idea as the sequential example in `02_workflows.ipynb`), then render the final string as Markdown.


In [ ]:
query = "How do you use the touchscreen to play music?"
config = {"configurable": {"thread_id": "day2-rag-demo"}}

final_answer = None
for step in rag_qa_workflow.stream(query, config=config, stream_mode="updates"):
    print(step)
    print()
    final_answer = next(iter(step.values()))

display(Markdown(final_answer))

## Next steps

- OpenAI embedding guide: [Embeddings](https://platform.openai.com/docs/guides/embeddings)
- OpenRouter models: [openrouter.ai/models](https://openrouter.ai/models)
- LangChain retrieval overview: [RAG](https://docs.langchain.com/oss/python/langchain/retrieval/)
